# PRABHĀSA — Kāraka Role-Contrast Supervision (H1_CONTRAST)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/prabhasa-babylm/blob/main/notebooks/demo_04_karaka_role_contrast.ipynb)

This notebook demonstrates the **kāraka role-contrast** method — the live research line
of PSALM, separate from the published Arm-B model in `demo_01`/`demo_02`.

**The idea, in one sentence.** Sanskrit grammar (Pāṇini) describes who-did-what-to-whom
with **kāraka** roles — *kartā* (the doer / agent), *karma* (the thing acted on / patient),
*karaṇa* (instrument), and so on. The method gives a small language model these roles as a
**gold training signal**: alongside ordinary masked-language-model pretraining, an auxiliary
head is asked to predict each token's kāraka role. The question is whether teaching argument
roles this way makes the model encode *who is the agent and who is the patient* better — and,
crucially, whether it learns this as an **abstract** fact that survives a change of surface
form (active ↔ passive), rather than a surface shortcut.

**What runs here (CPU-only, against a repo checkout):**
1. The **gold supervision** the method injects — a kāraka frame realized as both an active and
   a passive English sentence, each carrying the *same* gold roles.
2. The **specificity control** (shuffled roles) that makes the test honest.
3. The **probe instrument** and the construction-invariance test it powers.
4. The **committed findings** — reproduced from the result JSONs, with the honest conclusion.

The live encoder probe (extracting hidden states from a trained checkpoint) needs a GPU and the
27 checkpoints, which live in the project's AWS image — that step is shown but not executed here.

In [ ]:
# On Colab, uncomment to fetch the repo (the kāraka generators are in the repo, not on the Hub):
# !git clone -q https://github.com/SharathSPhD/prabhasa-babylm.git && pip install -q -e prabhasa-babylm

import sys
from pathlib import Path

# Locate the repo root whether this runs from the repo root or from notebooks/.
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents] if (p / "src" / "psalm").exists()), _here)
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "openspec/changes/karaka-role-contrast-supervision/results"
print("repo root :", ROOT)
print("results in:", RESULTS, "->", "found" if RESULTS.exists() else "MISSING")

## Part 1 — The gold supervision the method injects

A **kāraka frame** is a tiny meaning structure: a verb plus its participants, each tagged with
its role. The same frame is realized two ways — **active** ("X verbs Y") and **passive**
("Y is verbed by X"). The surface order changes, but the roles do **not**: the agent stays the
agent even when it slides into the "by …" phrase. These gold-by-construction labels are the
training signal for the auxiliary head — no parser, no noise.

(The lexicon recombines a small set of ~24 content words, so the sentences are structurally
varied but often semantically odd — e.g. "the rivers eat the forest". The diversity that matters
here is **structure, voice, number, and tense**, not vocabulary.)

In [ ]:
from psalm.domain.data.karaka_frames import enumerate_frames
from psalm.infrastructure.generators.english_frame_realizer import EnglishFrameRealizer

# WX role name -> (transliterated kāraka, plain-English gloss)
ROLE_GLOSS = {
    "karwA": ("kartā", "agent / doer"),
    "karma": ("karma", "patient / thing acted on"),
    "karaNam": ("karaṇa", "instrument"),
    "sampraxAnam": ("sampradāna", "recipient"),
    "apAxAnam": ("apādāna", "source"),
    "aXikaraNam": ("adhikaraṇa", "locus"),
    "kriyA": ("kriyā", "action / verb"),
    "separator": ("—", "function word"),
}

R = EnglishFrameRealizer()


def show_frame(fr):
    for voice in ("active", "passive"):
        sent = R.realize(fr, voice=voice)
        roles = R.word_roles(fr, voice=voice)
        print(f"\n{voice.upper():7}: {sent.text}")
        for word, wx in roles:
            if wx == "separator":
                continue
            k, gloss = ROLE_GLOSS.get(wx, (wx, "?"))
            print(f"          {word:10} → {k:11} ({gloss})")


# Find a few transitive frames (those with a karma/patient) for the active↔passive flip.
examples = []
for fr in enumerate_frames(200, seed=1):
    war = R.word_roles(fr, voice="active")
    if war and any(r == "karma" for _, r in war) and R.realize(fr, voice="passive"):
        examples.append(fr)
    if len(examples) == 3:
        break

for fr in examples:
    print("=" * 64)
    show_frame(fr)

### Visualizing the invariance

The same frame, two surfaces. Each token is colored by its gold kāraka role. Watch the
**kartā (agent)** and **karma (patient)** boxes: they swap *position* between active and passive,
but keep their *role color*. That invariance — role identity independent of surface slot — is
exactly the property the experiment later tests for inside the model.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

ROLE_COLOR = {
    "karwA": "#1f77b4",   # kartā / agent
    "karma": "#d62728",   # karma / patient
    "kriyA": "#2ca02c",   # kriyā / verb
    "aXikaraNam": "#9467bd",
    "karaNam": "#8c564b",
    "sampraxAnam": "#e377c2",
    "apAxAnam": "#ff7f0e",
    "separator": "#cccccc",
}

fr = examples[0]
fig, axes = plt.subplots(2, 1, figsize=(12, 2.6))
for ax, voice in zip(axes, ("active", "passive")):
    roles = R.word_roles(fr, voice=voice)
    x = 0.0
    for word, wx in roles:
        w = 0.10 + 0.075 * len(word)
        ax.add_patch(Rectangle((x, 0), w, 1, facecolor=ROLE_COLOR.get(wx, "#cccccc"),
                               edgecolor="white", alpha=0.85))
        ax.text(x + w / 2, 0.5, word, ha="center", va="center", color="white",
                fontsize=11, fontweight="bold")
        x += w + 0.02
    ax.set_xlim(0, x); ax.set_ylim(0, 1); ax.axis("off")
    ax.set_title(f"{voice.upper()}:  {R.realize(fr, voice=voice).text}", loc="left", fontsize=11)

handles = [Rectangle((0, 0), 1, 1, facecolor=ROLE_COLOR[k]) for k in ("karwA", "karma", "kriyA")]
fig.legend(handles, ["kartā (agent)", "karma (patient)", "kriyā (verb)"],
           loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.suptitle("Gold kāraka roles are invariant to the active↔passive surface flip", fontweight="bold")
plt.tight_layout()
plt.show()

## Part 2 — The specificity control (shuffled roles)

A model trained with *any* extra prediction target might improve a little, just from the
regularization. To isolate the effect of the **alignment** — the roles being attached to the
*right* tokens — the experiment includes a **shuffled-role control**: the same multiset of role
labels, randomly permuted so they no longer line up with their tokens. Any benefit that survives
shuffling is **not** alignment-specific. Three arms are trained: **baseline** (no aux),
**gold** (aligned roles), **shuffled** (permuted roles).

In [ ]:
from psalm.infrastructure.ml.contrast_corpus import shuffle_roles
from psalm.infrastructure.ml.shabdabodha_target import SHABDABODHA_LABELS

# The aux head's label set (id per role); 'separator' = function words / continuation pieces.
print("aux label set:", SHABDABODHA_LABELS)

# Gold per-word role ids for one active sentence, and the shuffled control.
fr = examples[0]
roles = R.word_roles(fr, voice="active")
gold_ids = [SHABDABODHA_LABELS.get(
    {"karwA": "karta", "karma": "karma", "kriyA": "kriya"}.get(wx, "separator"), 9)
    for _, wx in roles]
shuf_ids = shuffle_roles(gold_ids, seed=0)

words = [w for w, _ in roles]
print(f"\n{'token':10} {'gold role-id':>12} {'shuffled':>10}")
for w, g, s in zip(words, gold_ids, shuf_ids):
    print(f"{w:10} {g:>12} {s:>10}")
print("\nSame label multiset, broken alignment — the specificity control.")

## Part 3 — The probe instrument and the construction-invariance test

The model is a masked-LM; it does not *output* roles. To read what it encodes, freeze the
encoder and train a **linear probe** — a simple classifier — to predict each argument token's
role (agent vs patient) from its hidden vector. If the gold signal made roles more decodable,
the probe scores higher. Labels come from the **COGS** benchmark's logical forms (COGS is held
out from every arm, so the readout is fair).

The **crux** test sharpens this: train the probe on **active** sentences, test it on **passive**
ones. In active English the agent is the subject (early); in passive the *patient* is the subject
(early). A probe that learned a genuine role axis transfers; one that learned "subject = agent"
**inverts** on passives and scores *below* chance. This is the necessary condition for the roles
to be an abstraction rather than a surface trick.

In [ ]:
import re

ROLES = {"agent": 0, "theme": 1, "recipient": 2}
ROLE_RE = re.compile(r"[A-Za-z]+\.(agent|theme|recipient)\((?:x_\d+|[A-Za-z]+),(x_\d+|[A-Za-z]+)\)")


def parse_roles(sent, lf):
    # token-index -> role, from a COGS logical form (same logic as scripts/probe_invariance.py)
    toks = sent.split()
    out = {}
    for m in ROLE_RE.finditer(lf.replace(" ", "")):
        role, filler = m.group(1), m.group(2)
        idx = int(filler[2:]) if filler.startswith("x_") else (toks.index(filler) if filler in toks else -1)
        if 0 <= idx < len(toks):
            out[idx] = role
    return out


# Two real COGS lines (sentence, logical form): one active, one passive.
active = ("Jaxon squeezed the cookie in the bucket .",
          "* cookie ( x _ 3 ) ; * bucket ( x _ 6 ) ; squeeze . agent ( x _ 1 , Jaxon ) "
          "AND squeeze . theme ( x _ 1 , x _ 3 ) AND cookie . nmod . in ( x _ 3 , x _ 6 )")
passive = ("A cookie was blessed by Emma .",
           "cookie ( x _ 1 ) AND bless . theme ( x _ 3 , x _ 1 ) AND bless . agent ( x _ 3 , Emma )")

for label, (sent, lf) in [("ACTIVE", active), ("PASSIVE", passive)]:
    roles = parse_roles(sent, lf)
    toks = sent.split()
    print(f"\n{label}: {sent}")
    for i, t in enumerate(toks):
        if i in roles:
            tag = "subject(early)" if i <= 1 else "later"
            print(f"   [{i}] {t:9} = {roles[i]:7}  ({tag})")
print("\nActive: agent is the subject. Passive: the THEME is the subject.")
print("A 'subject = agent' probe therefore inverts on passives → below-chance accuracy.")

The live readout — loading a checkpoint and extracting hidden states — needs the GPU and the
trained checkpoints. The exact code is in `scripts/probe_invariance.py`; the core is:

```python
from psalm.infrastructure.ml.elc_trainer import load_elc_checkpoint
model, _ = load_elc_checkpoint(Path("runs/contrast_matrix/gold_s0/elc.pt"), device="cuda")
hid = model(torch.tensor([ids], device="cuda"))[1]["hidden_mlm"][0]   # per-token reps
# train nn.Linear(768, 3) on ACTIVE reps, evaluate on PASSIVE reps
```

We don't run it here; instead we reproduce its **committed results** below.

## Part 4 — The findings (reproduced from the committed result JSONs)

Three readouts, all agent-vs-theme accuracy (chance = 50%), 9 seeds × 3 arms:

- **within-construction** — train and test on the same (active) surface: does gold make roles
  more decodable at all?
- **active → passive (the crux)** — the construction-invariance transfer.
- **mixed → passive** — retrain the probe on *both* voices (so surface position no longer
  predicts role), then test on held-out passives: is there a position-robust role axis, and does
  gold encode it better?

In [ ]:
import json
import statistics as st

inv = json.load(open(RESULTS / "probe_invariance_results.json"))
mix = json.load(open(RESULTS / "probe_invariance_mixed_results.json"))
pr = json.load(open(RESULTS / "probe_results.json"))
ARMS, SEEDS = ["baseline", "gold", "shuffled"], range(9)


def mean(fn):
    return {a: round(st.mean(fn(a, s) for s in SEEDS), 2) for a in ARMS}


ceiling = mean(lambda a, s: inv[f"{a}_s{s}"]["active"]["acc_at"] * 100)   # active->active
crux = mean(lambda a, s: inv[f"{a}_s{s}"]["passive"]["acc_at"] * 100)     # active->passive
mixed = mean(lambda a, s: mix[f"{a}_s{s}"]["at_passive"] * 100)           # mixed->passive
within3 = mean(lambda a, s: pr[f"{a}_s{s}"]["acc3"] * 100)                # original 3-class probe

print(f"{'readout':28} {'baseline':>9} {'gold':>7} {'shuffled':>9}  gold-baseline")
for name, d in [("within-construction (3-class)", within3),
                ("active->active (ceiling)", ceiling),
                ("active->PASSIVE (crux)", crux),
                ("mixed->PASSIVE (invariant)", mixed)]:
    print(f"{name:28} {d['baseline']:>9} {d['gold']:>7} {d['shuffled']:>9}  {d['gold']-d['baseline']:+.2f}")

print(f"\nThe gold advantage attenuates as surface position is removed:")
print(f"  within-construction 3-class : +{within3['gold']-within3['baseline']:.1f}")
print(f"  position-robust (mixed pas) : +{mixed['gold']-mixed['baseline']:.1f}")

In [ ]:
import numpy as np

conds = ["active→active\n(ceiling)", "active→PASSIVE\n(crux)", "mixed→PASSIVE\n(invariant)"]
data = {a: [ceiling[a], crux[a], mixed[a]] for a in ARMS}
colors = {"baseline": "#7f7f7f", "gold": "#d62728", "shuffled": "#1f77b4"}

x = np.arange(len(conds)); w = 0.26
fig, ax = plt.subplots(figsize=(11, 5))
for i, a in enumerate(ARMS):
    ax.bar(x + (i - 1) * w, data[a], w, label=a, color=colors[a], alpha=0.9, edgecolor="black")
ax.axhline(50, ls="--", color="black", lw=1)
ax.text(2.45, 51.5, "chance (50%)", fontsize=9)
for j in range(len(conds)):  # annotate gold-baseline
    d = data["gold"][j] - data["baseline"][j]
    ax.annotate(f"gold−base {d:+.1f}", (x[j], max(data['baseline'][j], data['gold'][j]) + 3),
                ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(conds)
ax.set_ylabel("agent-vs-theme accuracy (%)"); ax.set_ylim(0, 105)
ax.set_title("Construction-invariance: the gold advantage is mostly surface-tied", fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

### What the figure says

- **Within a construction** the method works as advertised: gold makes argument roles markedly
  more decodable (the original probe: **+6.8** points, 3-class; +3.1 at the active→active ceiling
  above).
- **Across the active→passive flip the readout collapses *below* chance for every arm** — the
  active-trained role axis is dominated by surface position ("subject = agent"), so it inverts on
  passives. Gold is **no better** than baseline here (slightly worse). The strong
  construction-invariance crux **does not pass**.
- **Once the probe sees both voices** (mixed), a position-robust role axis is clearly there in
  every arm (~95%), and gold encodes it **+1.6** better — real, significant, and
  alignment-specific (it beats the shuffled control), but small.

**Conclusion (honest).** The method makes argument roles more decodable, but **mostly as a
surface/position-tied effect**; the genuinely construction-invariant residual is small (~1.6
points, well under the +3.0 a downstream task win would need). The crux gate is a STOP — the full
write-up, with the strongest counter-argument and the options from here, is in
`results/step0-construction-invariance-finding.md`.

## Summary

- **Method:** attach gold Pāṇinian kāraka (argument-role) labels to synthetic English and train
  an auxiliary head to predict them, alongside masked-LM pretraining. A shuffled-role arm
  controls for alignment; COGS provides a held-out probe.
- **Result:** roles become more linearly decodable **within a construction** (+6.8), but that is
  mostly **surface/position-tied** — it does not transfer across an active↔passive flip; only a
  small (~+1.6) position-robust residual is genuinely construction-invariant.
- **Reproduce the live probe:** `scripts/probe_invariance.py` and
  `scripts/probe_invariance_mixed.py` (need the GPU checkpoints in the project image).
- **Full finding + Tarka memo:** `results/step0-construction-invariance-finding.md`.